In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

In [0]:
%sql
SHOW TABLES IN TESIS

In [0]:
DB = "workspace.tesis"
TABLA_DESTINO = f"{DB}.enoe_jovenes_cdmx"

JOIN_KEYS = ["cd_a","ent","con","v_sel","n_hog","n_mud","n_ren","anio","trimestre"]

#Trimestres disponibles 

TRIMESTRES = [
    (2020, 1), (2020, 3), (2020, 4),
    (2021, 1), (2021, 2), (2021, 3), (2021, 4),
    (2022, 1), (2022, 2), (2022, 3), (2022, 4),
    (2023, 1), (2023, 2), (2023, 3), (2023, 4),
    (2024, 1), (2024, 2), (2024, 3), (2024, 4),
    (2025, 1), (2025, 2), (2025, 3),
]

#Columnas nuúmericas clave
# Se normalizarán con try_cast + trim para tolerar espacios y valores malformados
COLS_INT = [
    "cd_a", "ent", "con", "v_sel", "n_hog", "h_mud", "n_ren",
    "eda", "sex", "clase1", "clase2", "clase3",
    "pos_ocu", "rama_est1", "rama_est2", "seg_soc", "tip_con",
    "anios_esc", "niv_ins", "n_hij", "sub_o", "dur_des", "busqueda",
    "ur", "zona", "salario", "fac",
]
COLS_DBL = ["ingocup", "ing_x_hrs", "hrsocup"]

CENTINELAS = {
    "ingocup"   : [999999],
    "ing_x_hrs" : [999999.0],
    "hrsocup"   : [999, 99],
    "n_hij"     : [99],
    "anios_esc" : [99],
    "dur_des"   : [999],
}

print(f"Base de datos  : {DB}")
print(f"Tabla destino  : {TABLA_DESTINO}")
print(f"Trimestres     : {len(TRIMESTRES)}")

In [0]:
def normalizar_dataframe(df):
    """
    Normaliza un DataFrame leído desde los archivos del INEGI:
    - Recorta espacios en blanco en todas las columnas STRING
    - Castea columnas numéricas clave usando try_cast (no rompe el proceso)
    - Reemplaza valores centinela por null
    Retorna el DataFrame normalizado.
    """
    # Paso 1: trim en todas las columnas STRING para eliminar espacios en blanco
    # Este es el origen del error CAST_INVALID_INPUT que encontramos
    cols_string = [c for c, t in df.dtypes if t == "string"]
    for c in cols_string:
        df = df.withColumn(c, F.trim(F.col(c)))

    # Paso 2: casteo con try_cast para columnas enteras
    # try_cast devuelve null en lugar de romper el proceso
    for c in COLS_INT:
        if c in df.columns:
            df = df.withColumn(c, F.expr(f"try_cast({c} as INT)"))

    # Paso 3: casteo con try_cast para columnas dobles
    for c in COLS_DBL:
        if c in df.columns:
            df = df.withColumn(c, F.expr(f"try_cast({c} as DOUBLE)"))

    # Paso 4: reemplazar valores centinela por null
    for c, valores in CENTINELAS.items():
        if c in df.columns:
            df = df.withColumn(
                c,
                F.when(F.col(c).isin(valores), None).otherwise(F.col(c))
            )

    return df


def cargar_y_unir_tablas(prefijo, trimestres):
    """
    Carga tablas Delta de Unity Catalog con el patrón:
    workspace.tesis.{prefijo}_{anio}_t{trim}

    Une todas las tablas verticalmente con unionByName.
    Los trimestres que no existan se omiten sin interrumpir el proceso.
    Retorna el DataFrame unificado o None si no se cargó ninguna tabla.
    """
    lista = []
    fallidos = []

    for anio, trim in trimestres:
        nombre = f"{DB}.{prefijo}_{anio}_t{trim}"
        try:
            df_t = spark.table(nombre)
            lista.append(df_t)
            print(f"  OK   {nombre}")
        except Exception as e:
            fallidos.append(nombre)
            print(f"  FAIL {nombre} -> {str(e)[:60]}")

    if not lista:
        return None

    print(f"\n  Cargados : {len(lista)} | Fallidos : {len(fallidos)}")

    df = lista[0]
    for df_t in lista[1:]:
        # allowMissingColumns maneja columnas que el INEGI agrega entre años
        df = df.unionByName(df_t, allowMissingColumns=True)

    return df

In [0]:
print("=" * 60)
print("Cargando tablas SDEM...")
print("=" * 60)

sdem_full = cargar_y_unir_tablas("sdem", TRIMESTRES)

if sdem_full is None:
    raise Exception(
        "No se encontro ninguna tabla SDEM. "
        "Verifica que existen tablas con el patron: "
        + DB + ".sdem_{anio}_t{trim}"
    )

n_sdem = sdem_full.count()
n_cols_sdem = len(sdem_full.columns)
print("SDEM sin normalizar : " + str(n_sdem) + " registros | " + str(n_cols_sdem) + " columnas")

In [0]:
print("=" * 60)
print("Cargando tablas COE1...")
print("=" * 60)

coe1_full = cargar_y_unir_tablas("coe1", TRIMESTRES)

if coe1_full is None:
    raise Exception(
        "No se encontro ninguna tabla COE1. "
        "Verifica que existen tablas con el patron: "
        + DB + ".coe1_{anio}_t{trim}"
    )

n_coe1 = coe1_full.count()
n_cols_coe1 = len(coe1_full.columns)
print("COE1 sin normalizar : " + str(n_coe1) + " registros | " + str(n_cols_coe1) + " columnas")

In [0]:
print("Normalizando SDEM...")
sdem_full = normalizar_dataframe(sdem_full)
print("SDEM normalizado.")

print("\nNormalizando COE1...")
coe1_full = normalizar_dataframe(coe1_full)
print("COE1 normalizado.")

In [0]:
cols_sdem            = set(sdem_full.columns)
cols_coe1            = set(coe1_full.columns)
cols_nuevas_coe1     = sorted([c for c in cols_coe1 if c not in cols_sdem])

print(f"Columnas en SDEM                  : {len(cols_sdem)}")
print(f"Columnas en COE1                  : {len(cols_coe1)}")
print(f"Columnas nuevas aportadas por COE1: {len(cols_nuevas_coe1)}")
if cols_nuevas_coe1:
    print(f"  -> {cols_nuevas_coe1}")


# Verificar que las llaves de join existen en ambos DataFrames
llaves_en_sdem  = [k for k in JOIN_KEYS if k in cols_sdem]
llaves_en_coe1  = [k for k in JOIN_KEYS if k in cols_coe1]
llaves_faltantes = [k for k in JOIN_KEYS if k not in cols_sdem or k not in cols_coe1]

print(f"Llaves encontradas en SDEM : {llaves_en_sdem}")
print(f"Llaves encontradas en COE1 : {llaves_en_coe1}")

if llaves_faltantes:
    print(f"\nADVERTENCIA — Llaves faltantes: {llaves_faltantes}")
    print("El join se hará solo con las llaves disponibles en ambos DataFrames.")

# Usar solo las llaves que existen en ambos
llaves_validas = [k for k in JOIN_KEYS if k in cols_sdem and k in cols_coe1]
print(f"\nLlaves válidas para el join: {llaves_validas}")


if not llaves_validas:
    raise Exception(
        "No hay ninguna llave de join válida en común entre SDEM y COE1.\n"
        "Ejecuta sdem_full.printSchema() y coe1_full.printSchema() "
        "para verificar los nombres exactos de las columnas."
    )

if cols_nuevas_coe1:
    coe1_para_join = coe1_full.select(llaves_validas + cols_nuevas_coe1)
    df_enoe = sdem_full.join(coe1_para_join, on=llaves_validas, how="left")
    print(f"Join realizado con {len(llaves_validas)} llaves.")
else:
    df_enoe = sdem_full
    print("SDEM ya contiene todas las variables de COE1. Join omitido.")

print(f"Total registros : {df_enoe.count():,}")
print(f"Total columnas  : {len(df_enoe.columns)}")

In [0]:
print("Muestra de valores en 'ent' (primeras 10 entidades):")
df_enoe.select("ent").distinct().orderBy("ent").show(10)

print("Rango de valores en 'eda':")
df_enoe.select(
    F.min("eda").alias("edad_min"),
    F.max("eda").alias("edad_max"),
    F.count(F.when(F.col("eda").isNull(), 1)).alias("edad_nulos")
).show()

# COMMAND ----------

df_cdmx = df_enoe.filter(
    (F.col("ent") == 9)  &
    (F.col("eda") >= 18) &
    (F.col("eda") <= 29)
)

n_cdmx = df_cdmx.count()
print(f"Registros CDMX + 18-29 años : {n_cdmx:,}")

if n_cdmx == 0:
    raise Exception(
        "El filtro devolvió 0 registros.\n"
        "Revisa el diagnóstico de la celda anterior:\n"
        "  - ¿Aparece el valor 9 en la columna 'ent'?\n"
        "  - ¿El rango de 'eda' incluye valores entre 18 y 29?\n"
        "  - ¿Los valores de 'ent' y 'eda' son null después del casteo?"
    )

In [0]:
df_vars = (
    df_cdmx
    .withColumn(
        "desempleado",
        F.when(F.col("clase2") == 2, 1).otherwise(0)
    )
    .withColumn(
        "pea",
        F.when(F.col("clase1") == 1, 1).otherwise(0)
    )
    .withColumn(
        "grupo_edad",
        F.when(F.col("eda").between(18, 21), "18-21")
         .when(F.col("eda").between(22, 25), "22-25")
         .when(F.col("eda").between(26, 29), "26-29")
         .otherwise("otro")
    )
    .withColumn(
        "sexo_str",
        F.when(F.col("sex") == 1, "Hombre")
         .when(F.col("sex") == 2, "Mujer")
         .otherwise("No especificado")
    )
    .withColumn(
        "periodo_ord",
        (F.col("anio") - 2020) * 4 + F.col("trimestre")
    )
)

# Eliminar registros donde clase2 sea null
# (no podemos clasificar su condición de actividad)
n_antes   = df_vars.count()
df_vars   = df_vars.filter(F.col("clase2").isNotNull())
n_despues = df_vars.count()

print(f"Registros antes de filtrar nulos en clase2 : {n_antes:,}")
print(f"Registros eliminados                       : {n_antes - n_despues:,}")
print(f"Registros finales                          : {n_despues:,}")

In [0]:
COLUMNAS_FINALES = [
    # Periodo
    "anio", "trimestre", "periodo", "periodo_ord",
    # Sociodemográficas
    "eda", "sex", "sexo_str", "grupo_edad",
    "cs_p13_1",   # Nivel de escolaridad aprobado
    "anios_esc",  # Años de escolaridad acumulados
    "niv_ins",    # Nivel de instrucción resumido
    "e_con",      # Estado conyugal
    "n_hij",      # Número de hijos
    # Condición laboral
    "clase1",     # 1=PEA / 2=PNEA
    "clase2",     # 1=Ocupado / 2=Desocupado / 3=PNEA disp. / 4=PNEA no disp.
    "clase3",
    "pos_ocu",    # Posición en la ocupación
    "rama_est1",  # Rama de actividad económica
    "rama_est2",
    "hrsocup",    # Horas trabajadas
    "ingocup",    # Ingreso mensual en pesos
    "ing_x_hrs",  # Ingreso por hora
    "tip_con",    # Tipo de contrato
    "seg_soc",    # Seguridad social
    "sub_o",      # Subocupación
    "dur_des",    # Duración del desempleo en semanas
    "busqueda",   # Tipo de búsqueda de empleo
    # Variables construidas
    "desempleado",
    "pea",
    # Llaves de trazabilidad
    "cd_a", "ent", "con", "v_sel", "n_hog", "h_mud", "n_ren",
]

cols_ok       = [c for c in COLUMNAS_FINALES if c in df_vars.columns]
cols_ausentes = [c for c in COLUMNAS_FINALES if c not in df_vars.columns]

if cols_ausentes:
    print(f"Columnas no encontradas (se omiten): {cols_ausentes}")

df_maestro = df_vars.select(cols_ok)
print(f"\nDataFrame maestro: {df_maestro.count():,} filas x {len(df_maestro.columns)} columnas")

In [0]:
# Tasa de desempleo por periodo
tasa_periodo = (
    df_maestro
    .groupBy("periodo", "periodo_ord")
    .agg(
        F.count("*").alias("total"),
        F.sum("pea").alias("pea"),
        F.sum("desempleado").alias("desempleados"),
        F.round(F.mean("desempleado") * 100, 2).alias("tasa_pct")
    )
    .orderBy("periodo_ord")
)
display(tasa_periodo)

In [0]:
# Tasa por grupo de edad y sexo
tasa_edad_sexo = (
    df_maestro
    .groupBy("grupo_edad", "sexo_str")
    .agg(
        F.count("*").alias("total"),
        F.round(F.mean("desempleado") * 100, 2).alias("tasa_pct")
    )
    .orderBy("grupo_edad", "sexo_str")
)
display(tasa_edad_sexo)

In [0]:
# Porcentaje de nulos por columna
n_total = df_maestro.count()
nulos = df_maestro.select([
    F.round(
        F.sum(F.col(c).isNull().cast("int")) / n_total * 100, 2
    ).alias(c)
    for c in df_maestro.columns
])
display(nulos)